In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
from datasets import load_dataset
import numpy as np
import pandas as pd
import time

# ── Load Dataset ──────────────────────────────────────────────────────────
ds = load_dataset("bvsam/cic-ids-2017", "machine_learning")
df = ds["train"].to_pandas()

print("Label distribution:\n", df["Label"].value_counts())

# ── Sample & Clean ────────────────────────────────────────────────────────
df_sample = df.sample(frac=0.1, random_state=42).copy()

drop_cols = ["Flow ID", "Source IP", "Destination IP", "Timestamp"]
df_sample.drop(columns=drop_cols, errors="ignore", inplace=True)

# Replace inf and NaN
df_sample.replace([np.inf, -np.inf], np.nan, inplace=True)
df_sample.fillna(0, inplace=True)

# Handle single-sample classes
class_counts = df_sample["Label"].value_counts()
single_sample_classes = class_counts[class_counts <= 1].index
if not single_sample_classes.empty:
    print(f"Dropping single-sample classes: {list(single_sample_classes)}")
    df_sample = df_sample[~df_sample["Label"].isin(single_sample_classes)]

# Encode labels
le = LabelEncoder()
y = le.fit_transform(df_sample["Label"])
num_classes = len(le.classes_)
print(f"Classes ({num_classes}): {list(le.classes_)}")

# Scale features
X = df_sample.drop(columns=["Label"])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Feature count: {X_scaled.shape[1]}")

# ── Reshape features into 2D patches ─────────────────────────────────────
# ViT needs a 2D image — we reshape the feature vector into a square grid
# CIC-IDS2017 has ~70 features, pad to 100 → 10x10 image

IMG_SIZE = 10   # 10x10 = 100 pixels
N_FEATURES_PADDED = IMG_SIZE * IMG_SIZE

def features_to_image(X, img_size):
    """Pad or truncate feature vector and reshape into (img_size x img_size)"""
    n_samples, n_features = X.shape
    if n_features < img_size * img_size:
        # Pad with zeros
        padding = np.zeros((n_samples, img_size * img_size - n_features))
        X = np.concatenate([X, padding], axis=1)
    else:
        # Truncate
        X = X[:, :img_size * img_size]
    # Reshape to (n_samples, 1, img_size, img_size) — 1 channel like grayscale
    return X.reshape(n_samples, 1, img_size, img_size)

X_images = features_to_image(X_scaled, IMG_SIZE)
print(f"Image shape per sample: {X_images.shape[1:]}")  # should be (1, 10, 10)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_images, y, test_size=0.2, random_state=42, stratify=y
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def to_tensor(X, y):
    return torch.tensor(X, dtype=torch.float32).to(device), \
           torch.tensor(y, dtype=torch.long).to(device)

X_tr_t, y_tr_t = to_tensor(X_tr, y_tr)
X_te_t, y_te_t = to_tensor(X_te, y_te)

train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=256, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_te_t, y_te_t), batch_size=256, shuffle=False)

class PatchEmbedding(nn.Module):
    """Splits image into patches and linearly projects each patch"""
    def __init__(self, img_size, patch_size, in_channels, d_model):
        super().__init__()
        self.patch_size  = patch_size
        self.n_patches   = (img_size // patch_size) ** 2
        patch_dim        = in_channels * patch_size * patch_size

        # Linear projection of flattened patches (as described in tactic)
        self.projection  = nn.Linear(patch_dim, d_model)

        # Learnable [CLS] token and position embeddings
        self.cls_token   = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed   = nn.Parameter(torch.zeros(1, self.n_patches + 1, d_model))

    def forward(self, x):
        B, C, H, W  = x.shape
        p           = self.patch_size

        # Split image into patches: (B, n_patches, patch_dim)
        x = x.unfold(2, p, p).unfold(3, p, p)         # (B, C, H/p, W/p, p, p)
        x = x.contiguous().view(B, C, -1, p * p)       # (B, C, n_patches, p*p)
        x = x.permute(0, 2, 1, 3).contiguous()         # (B, n_patches, C, p*p)
        x = x.view(B, x.shape[1], -1)                  # (B, n_patches, patch_dim)

        x   = self.projection(x)                        # (B, n_patches, d_model)
        cls = self.cls_token.expand(B, -1, -1)          # (B, 1, d_model)
        x   = torch.cat([cls, x], dim=1)                # (B, n_patches+1, d_model)
        x   = x + self.pos_embed                        # add position embedding
        return x


class MLPHead(nn.Module):
    """MLP classification head as specified in tactic"""
    def __init__(self, d_model, n_classes, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, n_classes)
        )

    def forward(self, x):
        return self.net(x)


class ViT(nn.Module):
    def __init__(self, img_size=10, patch_size=2, in_channels=1,
                 d_model=64, n_heads=8, n_layers=6,
                 mlp_dim=128, dropout=0.1, n_classes=2):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = MLPHead(d_model, n_classes, hidden_dim=mlp_dim)

    def forward(self, x):
        x   = self.patch_embed(x)       # patch + position embedding
        x   = self.transformer(x)       # transformer encoder
        cls = self.norm(x[:, 0, :])     # CLS token
        return self.head(cls)           # MLP head → class


model = ViT(
    img_size=IMG_SIZE,
    patch_size=2,       # 2x2 patches → 25 patches from 10x10 image
    in_channels=1,
    d_model=64,
    n_heads=8,
    n_layers=6,         # deeper than Tactic 7 — ViT typically uses more layers
    mlp_dim=128,
    dropout=0.1,
    n_classes=num_classes
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {total_params:,}")

optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss()
EPOCHS = 15

print("\n── Training ViT ─────────────────────────────")
for epoch in range(EPOCHS):
    model.train()
    epoch_loss, correct, total = 0, 0, 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        logits = model(X_batch)
        loss   = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        correct    += (logits.argmax(1) == y_batch).sum().item()
        total      += y_batch.size(0)

    print(f"  Epoch {epoch+1:2d}/{EPOCHS} | "
          f"Loss: {epoch_loss/len(train_loader):.4f} | "
          f"Train Acc: {correct/total:.4f}")

# ── Evaluation ────────────────────────────────────────────────────────────
model.eval()
all_preds, all_labels = [], []

start = time.time()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        preds = model(X_batch).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.cpu().numpy())
vit_response_time = (time.time() - start) / len(X_te)

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

cm = confusion_matrix(all_labels, all_preds, labels=list(range(num_classes)))
acc = accuracy_score(all_labels, all_preds)

# ── Per-class metrics ─────────────────────────────────────────────────────
print(f"\n── Per-class Results ────────────────────────")
detection_rates, false_positive_rates = [], []

for i in range(num_classes):
    TP = cm[i, i]
    FN = cm[i, :].sum() - TP
    FP = cm[:, i].sum() - TP
    TN = cm.sum() - TP - FN - FP
    dr  = TP / (TP + FN) if (TP + FN) > 0 else 0
    fpr = FP / (FP + TN) if (FP + TN) > 0 else 0
    detection_rates.append(dr)
    false_positive_rates.append(fpr)
    print(f"  {le.classes_[i]:30s}  DR: {dr:.4f}  FPR: {fpr:.4f}")

# ── Summary ───────────────────────────────────────────────────────────────
print(f"\n══ Tactic 13: ViT Summary ═══════════════════")
print(f"{'Accuracy':<28} {acc:.4f}")
print(f"{'Macro Detection Rate':<28} {np.mean(detection_rates):.4f}")
print(f"{'Macro False Positive Rate':<28} {np.mean(false_positive_rates):.4f}")
print(f"{'Response Time (ms/sample)':<28} {vit_response_time*1000:.4f}")

Label distribution:
 Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64
Classes (15): ['BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Heartbleed', 'Infiltration', 'PortScan', 'SSH-Patator', 'Web Attack � Brute Force', 'Web Attack � Sql Injection', 'Web Attack � XSS']
Feature count: 78
Image shape per sample: (1, 10, 10)
Using device: cuda
Model parameters: 312,335

── Training ViT ─────

/tmp/ipykernel_1423/136236224.py:151: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


  Epoch  1/15 | Loss: 0.4251 | Train Acc: 0.8998
  Epoch  2/15 | Loss: 0.1366 | Train Acc: 0.9534
  Epoch  3/15 | Loss: 0.1084 | Train Acc: 0.9612
  Epoch  4/15 | Loss: 0.0918 | Train Acc: 0.9655
  Epoch  5/15 | Loss: 0.0829 | Train Acc: 0.9677
  Epoch  6/15 | Loss: 0.0755 | Train Acc: 0.9697
  Epoch  7/15 | Loss: 0.0708 | Train Acc: 0.9707
  Epoch  8/15 | Loss: 0.0661 | Train Acc: 0.9721
  Epoch  9/15 | Loss: 0.0635 | Train Acc: 0.9730
  Epoch 10/15 | Loss: 0.0612 | Train Acc: 0.9736
  Epoch 11/15 | Loss: 0.0596 | Train Acc: 0.9743
  Epoch 12/15 | Loss: 0.0577 | Train Acc: 0.9750
  Epoch 13/15 | Loss: 0.0565 | Train Acc: 0.9757
  Epoch 14/15 | Loss: 0.0549 | Train Acc: 0.9762
  Epoch 15/15 | Loss: 0.0544 | Train Acc: 0.9763

── Per-class Results ────────────────────────
  BENIGN                          DR: 0.9780  FPR: 0.0271
  Bot                             DR: 0.3846  FPR: 0.0000
  DDoS                            DR: 0.9860  FPR: 0.0002
  DoS GoldenEye                   DR: 0.9756